# Raynaud's CREST Pipeline
**Transcriptome-Guided Drug Repurposing for Raynaud's Phenomenon in Systemic Sclerosis**

Glen Ritschel | Ritschel Research | 2026

GitHub: https://github.com/glenritschel/raynaud-crest

**Before running:** Runtime > Change runtime type > T4 GPU

## Step 1: Clone repo and install dependencies

In [ ]:
import os
REPO_URL = "https://github.com/glenritschel/raynaud-crest"
REPO_DIR = "raynaud-crest"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}/notebooks
print("Working directory:", os.getcwd())


In [ ]:
import subprocess, sys
packages = ["scvi-tools==1.3.3", "scanpy==1.11.5", "gseapy==1.1.10", "leidenalg", "python-igraph", "scikit-misc", "biopython"]
for pkg in packages:
    print("Installing", pkg, "...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)
print("All dependencies installed.")


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")


## Step 2: Download raw data from NCBI GEO

In [ ]:
import os, urllib.request, glob

REPO_H5_DIR = "/content/raynaud-crest/data/raw/GSE138669"
os.makedirs(REPO_H5_DIR, exist_ok=True)

SAMPLES = [
    ("GSM4115868", "SC1"),  ("GSM4115869", "SC2"),
    ("GSM4115870", "SC4"),  ("GSM4115871", "SC5"),
    ("GSM4115872", "SC18"), ("GSM4115873", "SC19"),
    ("GSM4115874", "SC32"), ("GSM4115875", "SC33"),
    ("GSM4115876", "SC34"), ("GSM4115877", "SC49"),
    ("GSM4115878", "SC50"), ("GSM4115879", "SC60"),
    ("GSM4115880", "SC68"), ("GSM4115881", "SC69"),
    ("GSM4115882", "SC70"), ("GSM4115883", "SC86"),
    ("GSM4115884", "SC119"),("GSM4115885", "SC124"),
    ("GSM4115886", "SC125"),("GSM4115887", "SC185"),
    ("GSM4115888", "SC188"),("GSM4115889", "SC189"),
]

# Change SAMPLES[:2] to SAMPLES for the full 22-sample run
samples_to_get = SAMPLES

print("Downloading", len(samples_to_get), "samples from NCBI GEO...")
base = "https://ftp.ncbi.nlm.nih.gov/geo/samples"

for gsm, subj in samples_to_get:
    fname = gsm + "_" + subj + "raw_feature_bc_matrix.h5"
    dest = os.path.join(REPO_H5_DIR, fname)
    if os.path.exists(dest):
        print("  Already present:", fname)
        continue
    prefix = gsm[:7] + "nnn"
    url = base + "/" + prefix + "/" + gsm + "/suppl/" + fname
    print("  Downloading", fname, "...", end=" ", flush=True)
    try:
        urllib.request.urlretrieve(url, dest)
        print(round(os.path.getsize(dest) / 1e6, 1), "MB")
    except Exception as e:
        print("FAILED:", e)

downloaded = glob.glob(os.path.join(REPO_H5_DIR, "*.h5"))
print("Done.", len(downloaded), ".h5 files ready.")


## Step 3: Script 01 — Load, QC, Vascular Isolation

In [ ]:
%run ../src/01_load_qc.py


## Step 4: Script 02 — scVI Embedding + Leiden Clustering (GPU)

In [ ]:
%run ../src/02_scvi_embed.py


## Step 5: Script 03 — EC Subtype Annotation

In [ ]:
%run ../src/03_annotate_clusters.py


## Step 6: Script 04 — Raynaud's Signature Scoring

In [ ]:
%run ../src/04_signature_scoring.py


## Step 7: Script 05 — Differential Expression

In [ ]:
%run ../src/05_differential_expression.py


## Step 8: Script 06 — LINCS L1000 Reversal Scoring

In [ ]:
%run ../src/06_lincs_repurposing.py


## Step 9: Script 07 — Novelty Assessment + Priority Scoring

In [ ]:
%run ../src/07_novelty_prioritization.py


## Step 10: Save outputs to Google Drive

In [ ]:
from google.colab import drive
import shutil, os, glob
drive.mount("/content/drive")
DRIVE_OUTPUT = "/content/drive/MyDrive/Ritschel_Research/raynaud_crest_output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
outputs = glob.glob("/content/raynaud-crest/data/processed/*.csv") + glob.glob("/content/raynaud-crest/data/processed/*.h5ad")
for f in outputs:
    shutil.copy2(f, os.path.join(DRIVE_OUTPUT, os.path.basename(f)))
    print("Saved:", os.path.basename(f))
print("All outputs saved to", DRIVE_OUTPUT)
